# ⚙️ 02 — Data Preprocessing
> Notebook ini mencakup pembersihan data: hapus kolom Id, imputasi missing value,
> deteksi outlier, encoding kategorikal, dan simpan data terproses ke `data/processed/`.

## 📦 Import Library

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import os

from src.data_loader import load_config, load_raw_data
from src.preprocessing import (
    drop_id_column, split_by_target,
    fill_missing_values, detect_outliers_iqr, encode_categorical
)

print('✅ Library berhasil diimpor.')

## 📂 Muat Data Mentah

In [ ]:
cfg     = load_config('../config.yaml')
dataset = load_raw_data(cfg['data']['raw_path'])
dataset.head()

## 🗑️ 4.1 Hapus Kolom Id

In [ ]:
df = drop_id_column(dataset, id_col=cfg['data']['id_column'])
print(f'Kolom tersisa: {df.columns.tolist()}')

## ✂️ 4.2 Pisahkan Data Train / Test

In [ ]:
df_train, df_test = split_by_target(df, target=cfg['data']['target_column'])
print(f'Train shape : {df_train.shape}')
print(f'Test shape  : {df_test.shape}')

## 🩹 4.3 Imputasi Missing Values
- **Numerik** → diisi dengan **median** (robust terhadap outlier)
- **Kategorikal** → diisi dengan **modus**

In [ ]:
df_train = fill_missing_values(df_train, target=cfg['data']['target_column'])
df_train.isnull().sum()[df_train.isnull().sum() > 0]

## 📌 4.4 Deteksi Outlier (IQR)

In [ ]:
outliers, lower, upper = detect_outliers_iqr(
    df_train, col=cfg['data']['target_column']
)
print(f'ℹ️  Outlier dipertahankan = {cfg["preprocessing"]["keep_outliers"]}')
outliers.head()

## 🔠 4.5 Encoding Kategorikal (OneHotEncoding)

In [ ]:
cat_cols = df_train.select_dtypes(include=['object']).columns.tolist()
print(f'Kolom kategorikal: {cat_cols}')

df_final, encoder = encode_categorical(df_train, cat_cols)
df_final.head(3)

## 💾 4.6 Simpan Data Terproses

In [ ]:
os.makedirs('../data/processed', exist_ok=True)

train_path = cfg['data']['train_processed']
df_final.to_csv(f'../{train_path}', index=False)
print(f'✅ Data training disimpan : {train_path}')
print(f'   Shape                  : {df_final.shape}')

## ✅ Ringkasan Preprocessing

In [ ]:
print(f'Jumlah baris training : {len(df_final):,}')
print(f'Jumlah fitur          : {df_final.shape[1] - 1}')
print(f'Missing values tersisa: {df_final.isnull().sum().sum()}')
df_final.dtypes.value_counts()